In [1]:
import sys 
sys.path.append('/home/hydrogen/workspace/Space_GW/pespace')
sys.path.append('/home/hydrogen/workspace/Space_GW/wf4ti')

import taichi as ti
import numpy as np
import h5py
from matplotlib import pyplot as plt
%matplotlib inline

[Taichi] version 1.6.0, llvm 15.0.4, commit f1c6fbbd, linux, python 3.10.12


[I 07/11/24 18:53:12.506 1431774] [shell.py:_shell_pop_print@23] Graphical python shell detected, using wrapped sys.stdout


In [2]:
# file_path = "/home/hydrogen/workspace/Space_GW/LDC/LDC_data/LDC1-1_MBHB_v2_TD_9gc2s16.hdf5"
file_path = "/home/hydrogen/workspace/Space_GW/LDC/LDC_data/LDC1-1_MBHB_v1_1_TD.hdf5"
with h5py.File(file_path, "r") as f:
    # f.visit(lambda name: print(name))

    source_parameters = {}
    print('Source information: ')
    for parameter, value in f["H5LISA/GWSources/MBHB-0"].items():
        print(f"{parameter}: {value[()]}")
        source_parameters[parameter] = value[()]

    print(f["H5LISA/PreProcess/TDIGenerator"][()])
    TDI_data = f["H5LISA/PreProcess/TDIdata"][()]

Source information: 
Approximant: b'IMRPhenomD'
AzimuthalAngleOfSpin1: 0.6171792478977071
AzimuthalAngleOfSpin2: 4.75979656623224
Cadence: 10.0
CoalescenceTime: 25135000.0
Distance: 60.42017466175677
EclipticLatitude: -0.5256036732051035
EclipticLongitude: 1.1637
InitialAzimuthalAngleL: 0.30782038099413395
InitialPolarAngleL: 1.2498
Mass1: 2803843.776
Mass2: 285210.246
ObservationDuration: 41943040.0
PhaseAtCoalescence: 2.596553404898615
PolarAngleOfSpin1: 0.0
PolarAngleOfSpin2: 0.0
Redshift: 6.1178
Spin1: 0.8986046314480332
Spin2: 0.9465882372512956
hphcData: [[ 0.00000000e+00 -2.52087943e-22 -8.74551457e-21]
 [ 1.00000000e+01 -2.30277543e-22 -8.74164257e-21]
 [ 2.00000000e+01 -2.08463967e-22 -8.73765719e-21]
 ...
 [ 4.19430100e+07  0.00000000e+00  0.00000000e+00]
 [ 4.19430200e+07  0.00000000e+00  0.00000000e+00]
 [ 4.19430300e+07  0.00000000e+00  0.00000000e+00]]
b'X,Y,Z'


In [3]:
time_samples = TDI_data[:,0]
TDI_strain = TDI_data[:,1:]
print(TDI_strain)
print(TDI_strain.shape)
TDI_strain = TDI_strain.T
print(TDI_strain)
print(TDI_strain.shape)

duration = time_samples[-1]-time_samples[0]
cadence = time_samples[1]-time_samples[0]
print('duration: ', duration)
print('cadence: ', cadence)

[[ 1.43833880e-20  2.60735822e-20 -3.76029487e-20]
 [-4.00786160e-20 -1.10577887e-20  8.30232332e-20]
 [ 5.03369389e-20 -4.04076805e-21 -8.91464442e-20]
 ...
 [-9.93639790e-21  5.14937840e-21 -3.06222993e-21]
 [-4.46619576e-21  9.52535015e-21  3.00762453e-20]
 [-1.15332474e-21 -2.70341628e-20 -3.94004181e-22]]
(4194304, 3)
[[ 1.43833880e-20 -4.00786160e-20  5.03369389e-20 ... -9.93639790e-21
  -4.46619576e-21 -1.15332474e-21]
 [ 2.60735822e-20 -1.10577887e-20 -4.04076805e-21 ...  5.14937840e-21
   9.52535015e-21 -2.70341628e-20]
 [-3.76029487e-20  8.30232332e-20 -8.91464442e-20 ... -3.06222993e-21
   3.00762453e-20 -3.94004181e-22]]
(3, 4194304)
duration:  41943030.0
cadence:  10.0


In [4]:
from pespace.detectors import TDIChannelsData
ti.init()

/home/hydrogen/workspace/Space_GW/pespace/pespace/utilities.py:7: UserWarning: Wswiglal-redir-stdio:

SWIGLAL standard output/error redirection is enabled in IPython.
This may lead to performance penalties. To disable locally, use:

with lal.no_swig_redirect_standard_output_error():
    ...

To disable globally, use:

lal.swig_redirect_standard_output_error(False)

Note however that this will likely lead to error messages from
LAL functions being either misdirected or lost when called from
Jupyter notebooks.

To suppress this warning, use:

import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
import lal

  import lal


[Taichi] Starting on arch=x64


In [5]:
LDC_mbhb = TDIChannelsData()
LDC_mbhb.set_data_info(channels=('X', 'Y', 'Z'), 
                                               generation='1.5', 
                                               duration=duration,
                                               cadence=cadence,
                                               start_time=time_samples[0]
                                               )
LDC_mbhb.data_info
LDC_mbhb.set_time_domain_data_from_input_array(channels=('X', 'Y', 'Z'), 
                                               generation='1.5', 
                                               duration=duration,
                                               cadence=cadence,
                                               TDI_data=TDI_strain,
                                               start_time=time_samples[0])
print(LDC_mbhb.time_domain_TDI_data)
print(LDC_mbhb.time_samples)
# check the recovered array
print(np.sum(LDC_mbhb.time_samples.to_numpy()-time_samples))
for idx, chan in enumerate(('X', 'Y', 'Z')):
    print(np.sum(LDC_mbhb.time_domain_TDI_data.get_member_field(chan).to_numpy() - TDI_strain[idx]))

{'X': array([ 1.43833880e-20, -4.00786160e-20,  5.03369389e-20, ...,
       -9.93639790e-21, -4.46619576e-21, -1.15332474e-21]), 'Y': array([ 2.60735822e-20, -1.10577887e-20, -4.04076805e-21, ...,
        5.14937840e-21,  9.52535015e-21, -2.70341628e-20]), 'Z': array([-3.76029487e-20,  8.30232332e-20, -8.91464442e-20, ...,
       -3.06222993e-21,  3.00762453e-20, -3.94004181e-22])}
[1.000000e+01 2.000000e+01 3.000000e+01 ... 4.194302e+07 4.194303e+07
 4.194304e+07]
0.0
0.0
0.0
0.0


In [6]:
file_path = '/home/hydrogen/workspace/Space_GW/LDC/LDC_data/LDC1-1_MBHB_v2_FD_noiseless.hdf5.1'
with h5py.File(file_path, "r") as f:
    f.visit(lambda name: print(name))

    source_parameters = {}
    print('Source information: ')
    for parameter, value in f["H5LISA/GWSources/MBHB-0"].items():
        # print(f"{parameter}: {value[()]}")
        source_parameters[parameter] = value[()]

    # print(f["H5LISA/UserRequest/ConfigParams/Request"][()])
    TDI_data = f["H5LISA/PreProcess/TDIdata"][()]
    print(TDI_data)

H5LISA
H5LISA/Author
H5LISA/GWSources
H5LISA/GWSources/MBHB-0
H5LISA/GWSources/MBHB-0/Approximant
H5LISA/GWSources/MBHB-0/AzimuthalAngleOfSpin1
H5LISA/GWSources/MBHB-0/AzimuthalAngleOfSpin2
H5LISA/GWSources/MBHB-0/Cadence
H5LISA/GWSources/MBHB-0/CoalescenceTime
H5LISA/GWSources/MBHB-0/Distance
H5LISA/GWSources/MBHB-0/EclipticLatitude
H5LISA/GWSources/MBHB-0/EclipticLongitude
H5LISA/GWSources/MBHB-0/InitialAzimuthalAngleL
H5LISA/GWSources/MBHB-0/InitialPolarAngleL
H5LISA/GWSources/MBHB-0/Mass1
H5LISA/GWSources/MBHB-0/Mass2
H5LISA/GWSources/MBHB-0/ObservationDuration
H5LISA/GWSources/MBHB-0/PhaseAtCoalescence
H5LISA/GWSources/MBHB-0/PolarAngleOfSpin1
H5LISA/GWSources/MBHB-0/PolarAngleOfSpin2
H5LISA/GWSources/MBHB-0/Redshift
H5LISA/GWSources/MBHB-0/Spin1
H5LISA/GWSources/MBHB-0/Spin2
H5LISA/History
H5LISA/History/0000
H5LISA/History/0001
H5LISA/History/0002
H5LISA/History/0003
H5LISA/History/0004
H5LISA/History/0005
H5LISA/History/0006
H5LISA/PreProcess
H5LISA/PreProcess/TDIdata
H5LISA/Us

In [7]:
TDI_strain = TDI_data[:,1:4]
random_num = np.random.uniform(0.0, 2*np.pi, TDI_strain.shape)
TDI_strain = TDI_strain*np.exp(random_num*1j)
TDI_strain = TDI_strain.T
print(TDI_strain.shape)
TDI_strain

(3, 4194305)


array([[ 4.59187480e-26-9.78186199e-26j, -6.46819462e-26+8.70669823e-26j,
         1.07043546e-25-1.98637718e-26j, ...,
         2.77356966e-26+4.34196958e-26j, -1.22300980e-26-5.00182645e-26j,
         2.35066767e-26-4.57845379e-26j],
       [-1.05684698e-25-1.21182936e-25j,  1.48564108e-25+6.10722854e-26j,
         1.41171705e-25-7.62776076e-26j, ...,
         5.39309462e-27-2.94368758e-26j,  2.25928269e-26-1.96778638e-26j,
         1.71854746e-26+2.45840785e-26j],
       [-2.94685837e-26+4.37311480e-26j, -7.36826787e-27-5.16403888e-26j,
         8.79317313e-27-5.08350442e-26j, ...,
        -1.57309576e-26+1.47950487e-26j, -2.15153385e-26+8.18379606e-28j,
         1.66206767e-26+1.35928401e-26j]])

In [8]:
duration = source_parameters['ObservationDuration']
cadence = source_parameters['Cadence']
LDC_mbhb.set_data_info(channels=('X', 'Y', 'Z'), 
                        generation='1.5', 
                        duration=duration,
                        cadence=cadence
                        )
print(LDC_mbhb.data_info)
print(duration/cadence)
{'channels': ('X', 'Y', 'Z'), 'generation': '1.5', 'duration': 41943040.0, 'cadence': 10.0, 'start_time': 0.0, 'sampling_frequency': 0.1, 'delta_frequency': 2.384185791015625e-08, 'time_series_length': 4194305, 'frequency_series_length': 2096733, 'minimum_frequency': 1e-05, 'maximum_frequency': 0.05}
4194304.0


{'channels': ('X', 'Y', 'Z'), 'generation': '1.5', 'duration': 41943040.0, 'cadence': 10.0, 'start_time': 0.0, 'sampling_frequency': 0.1, 'delta_frequency': 2.384185791015625e-08, 'time_series_length': 4194305, 'frequency_series_length': 2096733, 'minimum_frequency': 1e-05, 'maximum_frequency': 0.05}
4194304.0


/home/hydrogen/workspace/Space_GW/pespace/pespace/detectors.py:300: UserWarning: You are setting data_info, whereas you have set TDI data of current instance previously.                            Setting `data_info` along may lead mismatch of data_info and the stored data. 
                            Please check whether this is intential.
  warnings.warn("You are setting data_info, whereas you have set TDI data of current instance previously. \


4194305

In [12]:
full_f_array = np.linspace(0.0, LDC_mbhb.data_info['sampling_frequency']/2, int((4194305+1)/2))
bound = (full_f_array>=LDC_mbhb.data_info['minimum_frequency']) * (full_f_array<=LDC_mbhb.data_info['maximum_frequency'])
full_f_array[bound].shape

(2096733,)

In [10]:
a = np.array([[0.1+0.2j, 0.3+0.4j], [0.5+0.6j, 0.6+0.7j]] )
print(a.real)
print(a.imag)
print((np.stack((a.real, a.imag), axis=-1)))

[[0.1 0.3]
 [0.5 0.6]]
[[0.2 0.4]
 [0.6 0.7]]
[[[0.1 0.2]
  [0.3 0.4]]

 [[0.5 0.6]
  [0.6 0.7]]]


In [24]:
a=np.array([0,1,2,3])
print(np.sum((a>=1.2).astype(np.int8)))

2
